# Step 1 — Fine-tuned Baselines: BERT and HateBERT on IHC

We fine-tune two pretrained models on the IHC (Implicit Hate Corpus) training set and evaluate them on the test set.

In [ ]:
# Uncomment if running in Colab
# !pip install -q transformers datasets accelerate scikit-learn

In [1]:
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings("ignore")

## 1. Load the Dataset

`SALT-NLP/implicit-hate` was removed from the Hub — `tasksource/implicit-hate-stg1` is a verified mirror of the same IHC corpus (ElSherief et al., 2021). It has a single `train` split with 21,480 tweets labelled as `not_hate`, `explicit_hate`, or `implicit_hate`.

Since there is no pre-made test split, we hold out 10% for evaluation. The binary label collapses explicit and implicit hate into a single **HS** class (label = 1); non-hate is **Non-HS** (label = 0). This is the standard setup for the IHC benchmark.

In [2]:
raw = load_dataset("tasksource/implicit-hate-stg1", split="train")
splits = raw.train_test_split(test_size=0.10, seed=42)

def add_binary_label(example):
    example["label"] = 0 if example["class"] == "not_hate" else 1
    return example

train_ds = splits["train"].map(add_binary_label)
test_ds  = splits["test"].map(add_binary_label)

print(f"Train: {len(train_ds):,}  Test: {len(test_ds):,}")
from collections import Counter
print("Train labels:", Counter(train_ds["label"]))
print("Test labels: ", Counter(test_ds["label"]))

Map:   0%|          | 0/19332 [00:00<?, ? examples/s]

Map:   0%|          | 0/2148 [00:00<?, ? examples/s]

Train: 19,332  Test: 2,148
Train labels: Counter({0: 11961, 1: 7371})
Test labels:  Counter({0: 1330, 1: 818})


## 2. Tokenization

Transformer models cannot work directly on raw text — they require a **tokenizer** that converts text into integer token IDs and an attention mask.

We truncate and pad every sequence to a fixed length of 128 tokens. Most tweets are well under this limit, so little information is lost. Each model has its own vocabulary and tokenizer, so we re-tokenize for each model.

In [3]:
def tokenize(ds, tokenizer, max_length=128):
    def _tok(batch):
        return tokenizer(
            batch["post"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
    return ds.map(_tok, batched=True)

## 3. Metrics

The `Trainer` calls `compute_metrics` after each evaluation epoch. We use **macro F1** as the primary metric — it averages F1 across both classes, penalising models that ignore the minority class (hate speech). Precision and recall give additional diagnostic information.

In [4]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1":  f1_score(labels, preds, average="macro",  zero_division=0),
        "macro_p":   precision_score(labels, preds, average="macro", zero_division=0),
        "macro_r":   recall_score(labels, preds, average="macro",    zero_division=0),
    }

## 4. Fine-tuning Loop

### What is a classification head?

A pretrained model like BERT has learned rich representations of text but has no built-in notion of hate speech. `AutoModelForSequenceClassification` adds a small **linear layer** (the classification head) on top of the `[CLS]` token representation, projecting it to 2 output logits (Non-HS, HS). This head is randomly initialised.

### What does fine-tuning do?

Fine-tuning runs gradient descent on the labelled IHC training set, updating **all** model weights — both the pretrained transformer layers and the new classification head. The pretrained weights are a warm start: the model has already learned grammar, semantics, and world knowledge, so it only needs a few epochs to adapt to the new task. Training from scratch would require far more data and compute.

In [5]:
MODELS = {
    "bert-base-uncased": "bert-base-uncased",
    "hateBERT":          "GroNLP/hateBERT",
}

results = {}

for model_name, model_id in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Fine-tuning: {model_name}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model     = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

    tok_train = tokenize(train_ds, tokenizer)
    tok_test  = tokenize(test_ds,  tokenizer)

    training_args = TrainingArguments(
        output_dir=f"./checkpoints/{model_name}",
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        report_to="none",
        seed=42,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tok_train,
        eval_dataset=tok_test,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # Final evaluation
    preds_output = trainer.predict(tok_test)
    preds  = np.argmax(preds_output.predictions, axis=-1)
    labels = test_ds["label"]

    print(classification_report(labels, preds, target_names=["Non-HS", "HS"]))

    results[model_name] = {
        "macro_f1":  f1_score(labels, preds, average="macro",  zero_division=0),
        "macro_p":   precision_score(labels, preds, average="macro", zero_division=0),
        "macro_r":   recall_score(labels, preds, average="macro",    zero_division=0),
    }


Fine-tuning: bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/19332 [00:00<?, ? examples/s]

Map:   0%|          | 0/2148 [00:00<?, ? examples/s]

...

BertForSequenceClassification LOAD REPORT from: GroNLP/hateBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/19332 [00:00<?, ? examples/s]

Map:   0%|          | 0/2148 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Macro F1,Macro P,Macro R
1,0.498427,0.428746,0.789765,0.795732,0.785537
2,0.353106,0.467005,0.782740,0.788575,0.778627
3,0.249919,0.544800,0.779168,0.785049,0.775054


              precision    recall  f1-score   support

      Non-HS       0.82      0.86      0.84      1330
          HS       0.75      0.69      0.72       818

    accuracy                           0.80      2148
   macro avg       0.79      0.78      0.78      2148
weighted avg       0.79      0.80      0.79      2148



## 5. Results

The table below compares both models on the IHC test set. Macro F1 is the headline number: it rewards models that perform well on both classes, not just the majority class. A higher F1 for HateBERT would suggest that domain-specific pretraining on hateful content gives a useful head start over generic BERT.

In [6]:
import pandas as pd

df = pd.DataFrame(results).T
df.columns = ["Macro F1", "Macro Precision", "Macro Recall"]
df.index.name = "Model"

df.style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold; background-color: #d4f1d4")

,Macro F1,Macro Precision,Macro Recall
Model,,,
bert-base-uncased,0.794,0.797,0.792
hateBERT,0.779,0.785,0.775
